In [51]:
# --------------------------------------------------
# 1. Imports (add these to the existing ones)
# --------------------------------------------------
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import itertools
from torchvision.transforms import Resize
from ast import literal_eval
from sklearn.metrics import matthews_corrcoef
from torchvision import models
from transformers import AutoTokenizer, AutoModelForMaskedLM
import math
import json
from collections import defaultdict
import seaborn as sns
from matplotlib.colors import LogNorm
import random
# --------------------------------------------------

In [52]:
with open('MCMFPP/train.txt') as f:
    lines = f.read().split('>')
    data = []
    for line in lines:
        if line.strip() != '':
            seq = line.split('\n')[1].strip()
            labels = [int(a) for a in line.split('\n')[0].strip()]
            data.append([seq]+ labels)

train_df = pd.DataFrame(data=data, columns=['sequence', 'AAP', 'ABP', 'ACP', 'ACVP', 'ADP', 'AEP', 'AFP', 'AHIVP',
       'AHP', 'AIP', 'AMRSAP', 'APP', 'ATP', 'AVP', 'BBP','BIP', 'CPP', 'DPPIP', 'QSP', 'SBP', 'THP'])
            
with open('MCMFPP/test.txt') as f:
    lines = f.read().split('>')
    data = []
    for line in lines:
        if line.strip() != '':
            seq = line.split('\n')[1].strip()
            labels = [int(a) for a in line.split('\n')[0].strip()]
            data.append([seq]+ labels)

test_df = pd.DataFrame(data=data, columns=['sequence', 'AAP', 'ABP', 'ACP', 'ACVP', 'ADP', 'AEP', 'AFP', 'AHIVP',
       'AHP', 'AIP', 'AMRSAP', 'APP', 'ATP', 'AVP', 'BBP','BIP', 'CPP', 'DPPIP', 'QSP', 'SBP', 'THP'])
    

# creating dataset

In [53]:
import torch
import numpy as np


import torch


# Example usage:
# seq = "ALDFR" -> Dipeptides: AL, LD, DF, FR
# vector = get_dipeptide_composition(seq)


esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")



PCP_columns = ['PCP_PC','PCP_NC','PCP_NE','PCP_PO','PCP_NP','PCP_AL','PCP_CY','PCP_AR','PCP_AC','PCP_BS','PCP_NE_pH','PCP_HB','PCP_HL','PCP_NT','PCP_HX','PCP_SC','PCP_SS_HE','PCP_SS_ST','PCP_SS_CO','PCP_SA_BU','PCP_SA_EX','PCP_SA_IN','PCP_TN','PCP_SM','PCP_LR','PCP_Z1','PCP_Z2','PCP_Z3','PCP_Z4','PCP_Z5']
PCP_DATA = {'A': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.24, -2.32, 0.6, -0.14, 1.3]),
'C': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.84, -1.67, 3.71, 0.18, -2.65]),
'D': np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 3.98, 0.93, 1.93, -2.46, 0.75]),
'E': np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 3.11, 0.26, -0.11, -3.04, -0.25]),
'F': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, -4.22, 1.94, 1.06, 0.54, -0.62]),
'G': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 2.05, -4.06, 0.36, -0.82, -0.38]),
'H': np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 2.47, 1.95, 0.26, 3.9, 0.09]),
'I': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, -3.89, -1.73, -1.71, -0.84, 0.26]),
'K': np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 2.29, 0.89, -2.49, 1.49, 0.31]),
'L': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, -4.28, -1.3, -1.49, -0.72, 0.84]),
'M': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, -2.85, -0.22, 0.47, 1.94, -0.98]),
'N': np.array([0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 3.05, 1.62, 1.04, -1.15, -1.61]),
'P': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, -1.66, 0.27, 1.84, 0.7, 2.0]),
'Q': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.75, 0.5, -1.44, -1.34, 0.66]),
'R': np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 3.52, 2.5, -3.5, 1.99, -0.17]),
'S': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 2.39, -1.07, 1.15, -1.39, 0.67]),
'T': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.75, -2.18, -1.12, -1.46, -0.4]),
'V': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, -2.59, -2.64, -1.54, -0.85, -0.02]),
'W': np.array([0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, -4.36, 3.94, 0.59, 3.44, -1.59]),
'Y': np.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, -2.54, 2.44, 0.43, 0.04, -1.47]),
'X': np.array([0.15, 0.1, 0.75, 0.25, 0.45, 0.3, 0.05, 0.15, 0.1, 0.15, 0.75, 0.5, 0.25, 0.3, 0.1, 0.1, 0.4, 0.35, 0.25, 0.4, 0.3, 0.3, 0.2, 0.45, 0.55, 0.0025000000000000356, 0.002499999999999991, 0.0019999999999999545, 0.0004999999999999877, -0.16299999999999998]),
'<pad>': np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
'<start>': np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
'<end>': np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
}

PCP_DATA_ID = dict()
for k, v in PCP_DATA.items():
    k = esm_tokenizer.token_to_id(k)
    PCP_DATA_ID[k] = v

def get_pcp_features(sequence):
    if isinstance(sequence, str):
        sequence = list(sequence)
        result = [PCP_DATA.get(c, PCP_DATA['X']) for c in sequence]
        result = torch.tensor(result, dtype=torch.float32)

    if isinstance(sequence, torch.Tensor):
        result = [PCP_DATA_ID.get(c, PCP_DATA_ID[3]) for c in sequence] # <unk>
        result = torch.tensor(result, dtype=torch.float32)
    return result



In [56]:
class PeptideDatasetBalanced(Dataset):
    """Pre-tokenizes all sequences at init time for much faster training."""

    def __init__(self, dataframe, tokenizer, label_columns, max_length=128):
        self.labels = dataframe[label_columns].values
        self.max_length = max_length
        
        # Pre-tokenize ALL sequences once
        sequences = dataframe['sequence'].tolist()
        encodings = tokenizer(
            sequences,
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        self.input_ids = encodings['input_ids']
        self.attention_masks = encodings['attention_mask']
        
        # Pre-compute PCP features for all sequences
        self.pcp_features = torch.stack([
            get_pcp_features(self.input_ids[i]) for i in range(len(sequences))
        ])
        
        print(f"  Pre-tokenized {len(sequences)} sequences")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            'input_ids': self.input_ids[index],
            'attention_mask': self.attention_masks[index],
            'pcp': self.pcp_features[index],
            'labels': torch.FloatTensor(self.labels[index])
        }


In [57]:
import random as pyrandom

# --- Data augmentation for single-function peptides (MCMFPP paper, Zhao et al. 2025) ---
def augment_single_function_peptides(df, label_columns, n_mask=2):
    """Augment single-function peptides by masking random amino acids with 'X'.
    For each single-function peptide, create n_mask masked copies.
    (Skipping flip/reverse since ESM2 expects natural-direction sequences.)
    """
    single_mask = df[label_columns].sum(axis=1) == 1
    single_df = df[single_mask]
    
    augmented_rows = []
    for _, row in single_df.iterrows():
        seq = row['sequence']
        labels = row[label_columns].values
        
        for _ in range(n_mask):
            if len(seq) > 1:
                pos = pyrandom.randint(0, len(seq) - 1)
                masked_seq = seq[:pos] + 'X' + seq[pos+1:]
                new_row = {'sequence': masked_seq}
                new_row.update({col: int(labels[i]) for i, col in enumerate(label_columns)})
                augmented_rows.append(new_row)
    
    if augmented_rows:
        aug_df = pd.DataFrame(augmented_rows)
        combined = pd.concat([df, aug_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
        print(f"  Augmented: {len(df)} → {len(combined)} ({len(single_df)} single-func × {n_mask} masks)")
        return combined
    return df

esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
LABEL_COLUMNS = ['AAP', 'ABP', 'ACP', 'ACVP', 'ADP', 'AEP', 'AFP', 'AHIVP',
       'AHP', 'AIP', 'AMRSAP', 'APP', 'ATP', 'AVP', 'BBP','BIP', 'CPP', 'DPPIP', 'QSP', 'SBP', 'THP']

MAX_LEN = 60

# Augment training data (paper optimal: n_mask=2)
train_df_aug = augment_single_function_peptides(train_df, LABEL_COLUMNS, n_mask=2)

train_dataset = PeptideDatasetBalanced(train_df_aug, esm_tokenizer, LABEL_COLUMNS, MAX_LEN)
test_dataset = PeptideDatasetBalanced(test_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN)

  Augmented: 7899 → 21335 (6718 single-func × 2 masks)
  Pre-tokenized 21335 sequences
  Pre-tokenized 1975 sequences


In [58]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [59]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoModelForMaskedLM

# =====================================================================
# Model v19 (Lite): Sequence-Level Cross-Attention + Task Query Decoder
# Parameter Count: ~19.6 Million
# =====================================================================

class ESM2_Encoder(nn.Module):
    def __init__(self, model_name, trainable=True, unfreeze_last_n=0):
        super().__init__()
        self.esm_mlm = AutoModelForMaskedLM.from_pretrained(model_name)
        self.hidden_size = self.esm_mlm.config.hidden_size
        if not trainable:
            for param in self.esm_mlm.parameters():
                param.requires_grad = False
            if unfreeze_last_n > 0:
                for layer in self.esm_mlm.esm.encoder.layer[-unfreeze_last_n:]:
                    for param in layer.parameters():
                        param.requires_grad = True

    def forward(self, input_ids, attention_mask):
        return self.esm_mlm.esm(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )
    def forward(self, x):
        w = x.mean(dim=-1)
        return x * self.fc(w).unsqueeze(-1)

class EnhancedCNN1D(nn.Module):
    def __init__(self, vocab_size=33, embed_dim=128, conv_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.branches = nn.ModuleList()
        for k in [3, 5, 7]:
            self.branches.append(nn.Sequential(
                nn.Conv1d(embed_dim, conv_dim, k, padding=k//2),
                nn.BatchNorm1d(conv_dim), nn.GELU(),
                nn.Conv1d(conv_dim, conv_dim, k, padding=k//2),
                nn.BatchNorm1d(conv_dim), nn.GELU(),
            ))
        self.res_proj = nn.Conv1d(embed_dim, conv_dim, 1)
        self.se = SEBlock(conv_dim * 3)
        self.dropout = nn.Dropout(0.2)
        self.hidden_size = conv_dim * 3 * 2  # 1536

    def forward(self, x):
        x = self.embed(x).transpose(1, 2)
        res = self.res_proj(x)
        outs = [branch(x) + res for branch in self.branches]
        combined = torch.cat(outs, dim=1)
        combined = self.se(combined)
        return self.dropout(torch.cat([combined.max(dim=-1).values, combined.mean(dim=-1)], dim=1))


class EnhancedPCP(nn.Module):
    """PCP encoder returning sequence-level features BxLx192."""
    def __init__(self, pcp_dim=30, conv_dim=64, hidden_size=96, num_layers=2):
        super().__init__()
        self.conv_branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(pcp_dim, conv_dim, k, padding=k//2),
                nn.BatchNorm1d(conv_dim), nn.GELU(),
            ) for k in [3, 5, 7]
        ])
        self.conv_merge = nn.Sequential(
            nn.Conv1d(conv_dim * 3, conv_dim * 2, 1),
            nn.BatchNorm1d(conv_dim * 2), nn.GELU(), nn.Dropout1d(0.1)
        )
        self.gru = nn.GRU(conv_dim * 2, hidden_size, num_layers=num_layers,
                          batch_first=True, bidirectional=True,
                          dropout=0.1 if num_layers > 1 else 0)
        self.output_dim = hidden_size * 2  # 192

    def forward(self, pcp_input, attention_mask):
        x = pcp_input.transpose(1, 2)
        x = torch.cat([b(x) for b in self.conv_branches], dim=1)
        x = self.conv_merge(x).transpose(1, 2)
        gru_out, _ = self.gru(x)
        return gru_out  # BxLx192



class MultiHeadAttentionPool(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.attn = nn.Sequential(
            nn.Linear(dim, 128), 
            nn.Tanh(), 
            nn.Linear(128, num_heads)
        )
        self._last_weights = None

    def forward(self, x, mask):
        scores = self.attn(x)
        scores = scores.masked_fill(mask.unsqueeze(-1) == 0, -1e4)
        weights = torch.softmax(scores, dim=1)
        self._last_weights = weights
        pooled = (x.unsqueeze(2) * weights.unsqueeze(-1)).sum(dim=1)
        return pooled.view(x.size(0), -1)

    # def orthogonality_loss(self):
    #     if self._last_weights is None:
    #         return 0.0
    #     w = self._last_weights.transpose(1, 2)
    #     gram = torch.bmm(w, w.transpose(1, 2))
    #     eye = torch.eye(self.num_heads, device=gram.device).unsqueeze(0)
    #     return ((gram - eye) ** 2).mean()

    def orthogonality_loss(self):
        """Penalize overlap between head attention distributions."""
        if self._last_weights is None:
            return 0.0
            
        # weights: [B, L, num_heads] -> transpose to [B, num_heads, L]
        w = self._last_weights.transpose(1, 2)  
        
        # Gram matrix of head attention distributions: shape [B, num_heads, num_heads]
        gram = torch.bmm(w, w.transpose(1, 2))  
        
        # Create a boolean mask for the off-diagonal elements (~torch.eye inverts the identity matrix)
        mask = ~torch.eye(self.num_heads, dtype=torch.bool, device=gram.device)
        
        # Penalize only the off-diagonal overlap to encourage heads to focus on different tokens.
        # We want these dot products to be pushed towards 0.
        loss = (gram[:, mask] ** 2).mean()
        
        return loss
    
class GatedFusion(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(input_dim, input_dim), nn.Sigmoid())
    def forward(self, x):
        return x * self.gate(x)

class PeptideNetwork(nn.Module):
    def __init__(self, num_classes=21, mask_token_id=32):
        super().__init__()
        self.mask_token_id = mask_token_id
        self.num_classes = num_classes

        # BOTH encoders are now the 8M parameter t6 model
        self.esm_t6_a = ESM2_Encoder("facebook/esm2_t6_8M_UR50D", trainable=True)        
        self.esm_t6_b = ESM2_Encoder("facebook/esm2_t6_8M_UR50D", trainable=False, unfreeze_last_n=2)                                  
        self.cnn = EnhancedCNN1D()          # Bx768
        #self.pcp_model = EnhancedPCP()      # BxLx192

        # REDUCED cross_dim to 128 to save parameters
        cross_dim = 128
        self.proj_t6_a = nn.Linear(320, cross_dim)
        self.proj_t6_b = nn.Linear(320, cross_dim) # Updated to 320 for t6
        #self.proj_pcp = nn.Linear(192, cross_dim)

        self.cross_t6_a = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.ln_ca_t6_a = nn.LayerNorm(cross_dim)
        
        self.cross_t6_b = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.ln_ca_t6_b = nn.LayerNorm(cross_dim)
        
        # self.cross_pcp = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        # self.ln_ca_pcp = nn.LayerNorm(cross_dim)

        self.pool_t6_a = MultiHeadAttentionPool(cross_dim, num_heads=4)
        self.pool_t6_b = MultiHeadAttentionPool(cross_dim, num_heads=4)
        # self.pool_pcp = MultiHeadAttentionPool(cross_dim, num_heads=4)

        # Bottleneck reduction layer before fusion
        # Concat size: 128*4*3 (pools) + 768 (CNN) = 1536 + 768 = 2304
        concat_size = cross_dim * 4 * 2 + self.cnn.hidden_size
        
        reduced_dim_size = 512
        self.dim_reduce = nn.Sequential(
            nn.Linear(concat_size, reduced_dim_size),
            nn.GELU()
        )
        
        # Fusion now operates efficiently on 512 dimensions
        self.fusion = GatedFusion(reduced_dim_size)
        self.ln = nn.LayerNorm(reduced_dim_size)

        # Task Query Decoder
        self.task_dim = 128
        self.n_memory_tokens = 16
        self.memory_proj = nn.Sequential(
            nn.Linear(reduced_dim_size, self.task_dim * self.n_memory_tokens),
            nn.GELU(),
        )
        self.task_queries = nn.Embedding(num_classes, self.task_dim)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=self.task_dim, nhead=4, dim_feedforward=256,
            batch_first=True, dropout=0.1
        )
        self.task_decoder = nn.TransformerDecoder(decoder_layer, num_layers=2)
        self.task_classifiers = nn.ModuleList([
            nn.Linear(self.task_dim, 1) for _ in range(num_classes)
        ])

    def _mask_tokens(self, input_ids, attention_mask, mask_prob=0.15):
        masked_ids = input_ids.clone()
        prob_matrix = torch.full_like(input_ids, mask_prob, dtype=torch.float)
        prob_matrix[attention_mask == 0] = 0
        prob_matrix[:, 0] = 0
        seq_lens = attention_mask.sum(dim=1)
        for i in range(len(seq_lens)):
            if seq_lens[i] > 1:
                prob_matrix[i, seq_lens[i] - 1] = 0
        mask = torch.bernoulli(prob_matrix).bool()
        masked_ids[mask] = self.mask_token_id
        return masked_ids

    def _extract_features(self, seq_input, seq_mask, pcp_input):
        esm6_a_seq = self.esm_t6_a(seq_input, seq_mask)       
        esm6_b_seq = self.esm_t6_b(seq_input, seq_mask)     
        cnn_feat = self.cnn(seq_input)                          

        t6_a = self.proj_t6_a(esm6_a_seq)       
        t6_b = self.proj_t6_b(esm6_b_seq)    


        kv_pad = seq_mask == 0  

        ca_t6_a, _ = self.cross_t6_a(t6_a, t6_b, t6_b, key_padding_mask=kv_pad)
        ca_t6_a = self.ln_ca_t6_a(t6_a + ca_t6_a)

        ca_t6_b, _ = self.cross_t6_b(t6_b, t6_a, t6_a, key_padding_mask=kv_pad)
        ca_t6_b = self.ln_ca_t6_b(t6_b + ca_t6_b)


        pooled_t6_a = self.pool_t6_a(ca_t6_a, seq_mask)
        pooled_t6_b = self.pool_t6_b(ca_t6_b, seq_mask)

        # Concat -> Reduce -> Fuse
        combined = torch.cat([pooled_t6_a, pooled_t6_b, cnn_feat], dim=1)  
        reduced = self.dim_reduce(combined)
        final_fusion = self.ln(reduced + self.fusion(reduced))
        
        return final_fusion

    def _classify(self, final_fusion):
        B = final_fusion.size(0)
        memory = self.memory_proj(final_fusion).view(B, self.n_memory_tokens, self.task_dim)
        tgt = self.task_queries.weight.unsqueeze(0).expand(B, -1, -1)
        decoded = self.task_decoder(tgt, memory)
        logits = torch.cat([self.task_classifiers[i](decoded[:, i, :])
                           for i in range(self.num_classes)], dim=1)
        return logits

    def forward(self, seq_input, seq_mask, pcp_input, mask_tokens=False):
        if mask_tokens and self.training:
            seq_input = self._mask_tokens(seq_input, seq_mask)
        final_fusion = self._extract_features(seq_input, seq_mask, pcp_input)
        return self._classify(final_fusion)

    def ortho_loss(self):
        return (self.pool_t6_a.orthogonality_loss() +
                self.pool_t6_b.orthogonality_loss() 
                ) / 2
    
    def get_features(self, seq_input, seq_mask, pcp_input, mask_tokens=False, mask_prob=0.15):
        if mask_tokens and self.training:
            seq_input = self._mask_tokens(seq_input, seq_mask, mask_prob)
        return self._extract_features(seq_input, seq_mask, pcp_input)

    def classify_features(self, combined):
        return self._classify(combined)

In [61]:
DEVICE    = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [62]:
DEVICE

'cuda:0'

In [63]:

FUNC_INDICES = [i for i in range(21)]
index_endpoint = {0: 'AAP',
1: 'ABP',
2: 'ACP',
3: 'ACVP',
4: 'ADP',
5: 'AEP',
6: 'AFP',
7: 'AHIVP',
8: 'AHP',
9: 'AIP',
10: 'AMRSAP',
11: 'APP',
12: 'ATP',
13: 'AVP',
14: 'BBP',
15: 'BIP',
16: 'CPP',
17: 'DPPIP',
18: 'QSP',
19: 'SBP',
20: 'THP',}

In [64]:
# pos_weight no longer needed - using AsymmetricLoss instead


In [ ]:
import copy
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        # Update parameters (weights & biases)
        for s_param, m_param in zip(self.shadow.parameters(), model.parameters()):
            s_param.data.mul_(self.decay).add_(m_param.data, alpha=1 - self.decay)
        
        # FIX: Update only floating point buffers (skip int buffers like num_batches_tracked)
        for s_buf, m_buf in zip(self.shadow.buffers(), model.buffers()):
            if s_buf.dtype.is_floating_point:
                s_buf.data.mul_(self.decay).add_(m_buf.data, alpha=1 - self.decay)
            else:
                # For integer buffers, just copy directly (no EMA smoothing needed)
                s_buf.data.copy_(m_buf.data)

    def forward(self, *args, **kwargs):
        return self.shadow(*args, **kwargs)

In [173]:
torch.cuda.empty_cache()

model = PeptideNetwork(num_classes=21, mask_token_id=32).to(DEVICE)
ema = EMA(model, decay=0.999)

# --- ASL: Asymmetric Loss for Multi-Label Classification ---
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        pos_part = targets * torch.log(p.clamp(min=1e-8))
        neg_p = (1 - p).clamp(min=1e-8)
        if self.clip > 0:
            neg_p = (neg_p + self.clip).clamp(max=1)
        neg_part = (1 - targets) * torch.log(neg_p)
        pos_weight = (1 - p) ** self.gamma_pos
        neg_weight = p ** self.gamma_neg
        loss = -(pos_weight * pos_part + neg_weight * neg_part)
        return loss.mean()

criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05)

# Optimizer — 3 LR groups (Lowered LRs to prevent early overfitting)
esm_t6_ids = set(id(p) for p in model.esm_t6_a.esm_mlm.parameters())
esm_t6_b_ids = set(id(p) for p in model.esm_t6_b.esm_mlm.parameters())

esm_t6_params = [p for p in model.parameters() if id(p) in esm_t6_ids and p.requires_grad]
esm_t6_b_params = [p for p in model.parameters() if id(p) in esm_t6_b_ids and p.requires_grad]
other_params = [p for p in model.parameters() if id(p) not in esm_t6_ids and id(p) not in esm_t6_b_ids and p.requires_grad]

optimizer = optim.AdamW([
    {'params': esm_t6_params, 'lr': 1e-5},     # Reduced from 1e-5
    {'params': esm_t6_b_params, 'lr': 2e-6},   # Reduced from 2e-6
    {'params': other_params, 'lr': 1e-4},      # Reduced from 1e-4
],  weight_decay=0.05)   

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {total_params/1e6:.1f}M | Trainable: {trainable_params/1e6:.1f}M")
print(f"Loss: ASL(γ-=4, γ+=1, clip=0.05) | v19: Triple CrossAttn + Task Query Decoder")
torch.cuda.empty_cache()

Total: 18.5M | Trainable: 13.5M
Loss: ASL(γ-=4, γ+=1, clip=0.05) | v19: Triple CrossAttn + Task Query Decoder


In [174]:
epochs = 80

In [175]:
# Warmup + Cosine Annealing
steps_per_epoch = len(train_loader)
warmup_epochs = 3
total_steps = epochs * steps_per_epoch

def lr_lambda(step):
    if step < warmup_epochs * steps_per_epoch:
        return step / (warmup_epochs * steps_per_epoch)
    progress = (step - warmup_epochs * steps_per_epoch) / (total_steps - warmup_epochs * steps_per_epoch)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(f"Warmup ({warmup_epochs} ep) + CosineAnnealing, total {epochs} epochs, {total_steps} steps")

Warmup (3 ep) + CosineAnnealing, total 80 epochs, 26720 steps


In [176]:
from sklearn.metrics import matthews_corrcoef, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import json

In [177]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.metrics import confusion_matrix, matthews_corrcoef, roc_curve, auc, f1_score
from tqdm import tqdm

# =====================================================================
# VECTORIZED METRIC FUNCTIONS (100x faster than loop-based versions)
# =====================================================================

def Aiming(y_hat, y):
    """Precision-like: avg ratio of correctly predicted labels over predicted labels."""
    intersection = np.sum(y_hat * y, axis=1)
    predicted = np.sum(y_hat, axis=1)
    # Only count samples with at least one intersection
    mask = intersection > 0
    scores = np.zeros(len(y_hat))
    scores[mask] = intersection[mask] / predicted[mask]
    return np.mean(scores)

def Coverage(y_hat, y):
    """Recall-like: avg ratio of correctly predicted labels over true labels."""
    intersection = np.sum(y_hat * y, axis=1)
    true_count = np.sum(y, axis=1)
    mask = intersection > 0
    scores = np.zeros(len(y_hat))
    scores[mask] = intersection[mask] / true_count[mask]
    return np.mean(scores)

def Accuracy(y_hat, y):
    """Jaccard-like: avg ratio of intersection over union per sample."""
    intersection = np.sum(y_hat * y, axis=1)
    union = np.sum(((y_hat + y) > 0).astype(float), axis=1)
    mask = intersection > 0
    scores = np.zeros(len(y_hat))
    scores[mask] = intersection[mask] / union[mask]
    return np.mean(scores)

def AbsoluteTrue(y_hat, y):
    """Exact match ratio."""
    return np.mean(np.all(y_hat == y, axis=1).astype(float))

def AbsoluteFalse(y_hat, y):
    """Hamming loss."""
    union = np.sum(((y_hat + y) > 0).astype(float), axis=1)
    intersection = np.sum(y_hat * y, axis=1)
    m = y.shape[1]
    return np.mean((union - intersection) / m)

def evaluate(score_label, y, threshold1=0.6, threshold2=0.4):
    _, m = y.shape
    if isinstance(threshold1, (int, float)):
        threshold1 = [threshold1] * m
    threshold1 = np.array(threshold1)
    
    y_hat = np.copy(score_label)
    
    # Vectorized thresholding
    max_vals = np.max(y_hat, axis=1)
    max_idxs = np.argmax(y_hat, axis=1)
    
    # Apply per-class thresholds
    y_hat_binary = (y_hat >= threshold1[np.newaxis, :]).astype(float)
    
    # For samples with no predictions above threshold, assign the max if > threshold2
    no_pred_mask = (y_hat_binary.sum(axis=1) == 0) & (max_vals > threshold2)
    y_hat_binary[no_pred_mask, max_idxs[no_pred_mask]] = 1.0
    
    aiming = Aiming(y_hat_binary, y)
    coverage = Coverage(y_hat_binary, y)
    accuracy = Accuracy(y_hat_binary, y)
    absolute_true = AbsoluteTrue(y_hat_binary, y)
    absolute_false = AbsoluteFalse(y_hat_binary, y)
    return aiming, coverage, accuracy, absolute_true, absolute_false

print("Vectorized metrics loaded.")


Vectorized metrics loaded.


In [180]:
# --- HELPER FUNCTIONS ---
def calculate_complete_metrics(y_true, y_pred_probs, thresholds, class_names):
    results = {}
    for i in range(y_true.shape[1]):
        y_pred_cls = (y_pred_probs[:, i] >= thresholds[i]).astype(int)
        mcc = matthews_corrcoef(y_true[:, i], y_pred_cls)
        acc = accuracy_score(y_true[:, i], y_pred_cls)
        prec = precision_score(y_true[:, i], y_pred_cls, zero_division=0)
        rec = recall_score(y_true[:, i], y_pred_cls, zero_division=0)
        f1 = f1_score(y_true[:, i], y_pred_cls, zero_division=0)
        try: auc_val = roc_auc_score(y_true[:, i], y_pred_probs[:, i])
        except ValueError: auc_val = 0.5
        tn, fp, fn, tp = confusion_matrix(y_true[:, i], y_pred_cls, labels=[0,1]).ravel()
        results[class_names[i]] = {
            "mcc": float(mcc), "threshold": float(thresholds[i]),
            "accuracy": float(acc), "precision": float(prec),
            "recall": float(rec), "f1_score": float(f1), "roc_auc": float(auc_val),
            "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)
        }
    return results

def find_optimal_thresholds(y_true, y_pred_probs):
    optimal = []
    for i in range(y_true.shape[1]):
        best_f1, best_t = -1, 0.5
        for t in np.arange(0.15, 0.85, 0.02):
            pred = (y_pred_probs[:, i] >= t).astype(int)
            f = f1_score(y_true[:, i], pred, zero_division=0)
            if f > best_f1: best_f1 = f; best_t = t
        optimal.append(best_t)
    return optimal

def find_optimal_thresholds_accuracy(y_true, y_pred_probs, rounds=4, step=0.02):
    thresholds = find_optimal_thresholds(y_true, y_pred_probs)
    best_acc = evaluate(y_pred_probs, y_true, threshold1=thresholds, threshold2=0.3)[2]
    for r in range(rounds):
        for i in range(y_true.shape[1]):
            best_t = thresholds[i]
            for t in np.arange(0.05, 0.95, step):
                trial = list(thresholds)
                trial[i] = t
                _, _, acc, _, _ = evaluate(y_pred_probs, y_true, threshold1=trial, threshold2=0.3)
                if acc > best_acc: best_acc = acc; best_t = t
            thresholds[i] = best_t
    return thresholds

def calculate_mcc(tp, tn, fp, fn):
    num = (tp * tn) - (fp * fn)
    den_sq = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    return num / math.sqrt(den_sq) if den_sq > 0 else 0.0

def feature_mixup(features, labels, alpha=0.4):
    batch_size = features.size(0)
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(batch_size, device=features.device)
    mixed_features = lam * features + (1 - lam) * features[index]
    mixed_labels = lam * labels + (1 - lam) * labels[index]
    return mixed_features, mixed_labels

def rdrop_kl_loss(logits1, logits2):
    kl1 = F.binary_cross_entropy_with_logits(logits1, torch.sigmoid(logits2).detach(), reduction='mean')
    kl2 = F.binary_cross_entropy_with_logits(logits2, torch.sigmoid(logits1).detach(), reduction='mean')
    return (kl1 + kl2) / 2

def enable_dropout(model):
    """
    Selectively enables dropout-related layers for Test-Time Augmentation (TTA)
    while keeping BatchNorm layers strictly frozen in eval() mode.
    """
    for module in model.modules():
        # 1. Catch all standard PyTorch dropouts (Dropout, Dropout1d, Dropout2d, etc.)
        # This will also naturally catch the Dropouts inside the Hugging Face ESM models
        if isinstance(module, nn.modules.dropout._DropoutNd):
            module.train()
            
        # 2. Catch complex modules that handle their own internal dropout
        elif isinstance(module, (
            nn.MultiheadAttention, 
            nn.TransformerDecoderLayer, 
            nn.TransformerDecoder, 
            nn.GRU
        )):
            module.train()
            
        # 3. Fallback catch just in case Hugging Face updates their backend 
        # to use a custom dropout class name that doesn't inherit from _DropoutNd
        elif "Dropout" in module.__class__.__name__:
            module.train()


# --- SWA ---
swa_start_epoch = 30
swa_model = None
swa_n = 0

def update_swa(model_state, swa_state, n):
    if swa_state is None:
        return {k: v.clone() for k, v in model_state.items()}, 1
    for k in swa_state:
        swa_state[k] = (swa_state[k] * n + model_state[k]) / (n + 1)
    return swa_state, n + 1

# --- MAIN TRAINING LOOP (v19: Triple CrossAttn + Task Query Decoder + ASL) ---
history_log = []
history_log_path = "mcmfpp.json"
best_model_path = "mcmfpp.pt"
best_accuracy = 0.0
scaler = torch.amp.GradScaler('cuda')
patience_counter = 0
early_stop_patience = 20
mixup_alpha = 0.4
rdrop_alpha = 1.0
tta_passes = 5

print("Training v19: Triple CrossAttn + Task Query Decoder + ASL")
print(f"  ASL(γ-=4, γ+=1, clip=0.05)")
print(f"  R-Drop α={rdrop_alpha} | Mixup α={mixup_alpha} | SWA from ep {swa_start_epoch}")
print(f"  Test eval: 100% | TTA={tta_passes}")
print("=" * 70)

for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0
    spec_y_train, spec_pred_train = [], []

    for data in train_loader:
        input_ids = data['input_ids'].to(DEVICE)
        attention_mask = data['attention_mask'].to(DEVICE)
        pcp = data['pcp'].to(DEVICE)
        labels = data['labels'].to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            # Two forward passes for R-Drop
            features1 = model.get_features(input_ids, attention_mask, pcp, mask_tokens=True, mask_prob=0.15)
            features2 = model.get_features(input_ids, attention_mask, pcp, mask_tokens=True, mask_prob=0.15)

            logits1_clean = model.classify_features(features1)
            logits2_clean = model.classify_features(features2)

            # R-Drop consistency
            rdrop_loss = rdrop_kl_loss(logits1_clean, logits2_clean)

            # Mixup on first pass
            mixed_features, mixed_labels = feature_mixup(features1, labels, alpha=mixup_alpha)
            logit_mixed =  model.classify_features(mixed_features)

            # ASL loss
            asl_loss1 = criterion(logit_mixed, mixed_labels)
            asl_loss2 = criterion(logits2_clean, labels)

            

            # Orthogonality loss on multi-head attention pools
            ortho = model.ortho_loss()

            loss = (asl_loss1 + asl_loss2) / 2 + rdrop_alpha * rdrop_loss + 0.1 * ortho

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema.update(model)

        total_train_loss += loss.item() * input_ids.size(0)


        spec_y_train.append(labels.cpu().numpy())
        spec_pred_train.append(torch.sigmoid(logits1_clean).detach().cpu().numpy())


    y_true_train = np.concatenate(spec_y_train)
    y_pred_train = np.concatenate(spec_pred_train)
    func_names = [index_endpoint[i] for i in FUNC_INDICES]

    train_thresh = [0.6]*21 #find_optimal_thresholds_accuracy(y_true_train, y_pred_train, rounds=4, step=0.01)

    # SWA
    if epoch >= swa_start_epoch:
        swa_model, swa_n = update_swa(ema.shadow.state_dict(), swa_model, swa_n)

    # --- EVAL ---
    if swa_model is not None and epoch >= swa_start_epoch:
        eval_model = copy.deepcopy(ema.shadow)
        eval_model.load_state_dict(swa_model)
        eval_model.eval()
    else:
        eval_model = ema.shadow
        eval_model.eval()

    total_test_loss = 0
    spec_y_test = []
    tta_preds = [[] for _ in range(tta_passes)]

    with torch.no_grad():
        for data in test_loader:
            input_ids = data['input_ids'].to(DEVICE)
            attention_mask = data['attention_mask'].to(DEVICE)
            pcp = data['pcp'].to(DEVICE)
            labels = data['labels'].to(DEVICE)

            with torch.amp.autocast('cuda'):
                logits = eval_model(input_ids, attention_mask, pcp)
                loss_val = criterion(logits, labels)
            total_test_loss += loss_val.item() * input_ids.size(0)
            spec_y_test.append(labels.cpu().numpy())
            tta_preds[0].append(torch.sigmoid(logits).detach().cpu().numpy())

            enable_dropout(eval_model)
            for t in range(1, tta_passes):
                with torch.amp.autocast('cuda'):
                    logits_t = eval_model(input_ids, attention_mask, pcp)
                tta_preds[t].append(torch.sigmoid(logits_t).detach().cpu().numpy())
            eval_model.eval()

    y_true_spec_test = np.concatenate(spec_y_test)
    tta_preds = [
        np.concatenate(tta_preds[t]) for t in range(tta_passes)
    ]
    y_pred_spec_test = np.mean(tta_preds, axis=0)
    
    # # Random 80% sampling
    # n_test = len(y_true_spec_test)
    # n_sample = int(n_test * test_sample_ratio)
    # sample_idx = np.random.choice(n_test, size=n_sample, replace=False)
    # y_true_spec_test = y_true_spec_test[sample_idx]
    # y_pred_spec_test = y_pred_spec_test[sample_idx]

    test_spec_metrics = calculate_complete_metrics(
        y_true_spec_test, y_pred_spec_test, train_thresh, func_names
    )

    avg_train_loss = total_train_loss / len(train_loader.dataset)
    avg_test_loss = total_test_loss / len(test_loader.dataset)

    tp, tn, fp, fn = 0, 0, 0, 0
    for k, v in test_spec_metrics.items():
        tp += v['tp']; tn += v['tn']; fp += v['fp']; fn += v['fn']
    total_mcc = calculate_mcc(tp, tn, fp, fn)

    aiming, coverage, accuracy, absolute_true, absolute_false = evaluate(
        y_pred_spec_test, y_true_spec_test, threshold1=train_thresh, threshold2=0.3
    )

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        patience_counter = 0
        torch.save(eval_model.state_dict(), best_model_path)
        marker = " >>> BEST <<<"
    else:
        patience_counter += 1
        marker = ""

    lr_now = optimizer.param_groups[0]['lr']
    swa_tag = f" [SWA n={swa_n}]" if epoch >= swa_start_epoch else ""
    print(f"Ep {epoch:02d}/{epochs} | LR: {lr_now:.1e} | Train: {avg_train_loss:.4f} | Test: {avg_test_loss:.4f} | MCC: {total_mcc:.4f}{swa_tag}")
    print(f"  Aim: {aiming:.4f} | Cov: {coverage:.4f} | Acc: {accuracy:.4f} | AbsT: {absolute_true:.4f} | Pat: {patience_counter}/{early_stop_patience}{marker}")
    print("-" * 70)

    epoch_data = {
        "epoch": epoch, "total_test_mcc": total_mcc,
        "train_loss": avg_train_loss, "test_loss": avg_test_loss,
        "aiming": float(aiming), "coverage": float(coverage),
        "accuracy": float(accuracy), "absolute_true": float(absolute_true),
        "absolute_false": float(absolute_false),
        "thresholds_train": train_thresh,
        "test_metrics": test_spec_metrics

    }
    history_log.append(epoch_data)
    with open(history_log_path, "w") as f:
        json.dump(history_log, f, indent=2)

    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping at epoch {epoch}!")
        break

    if swa_model is not None and epoch >= swa_start_epoch:
        del eval_model

print(f"\nTraining Complete! Best Accuracy: {best_accuracy:.4f}")

Training v19: Triple CrossAttn + Task Query Decoder + ASL
  ASL(γ-=4, γ+=1, clip=0.05)
  R-Drop α=1.0 | Mixup α=0.4 | SWA from ep 30
  Test eval: 100% | TTA=5
Ep 01/80 | LR: 3.3e-06 | Train: 0.6943 | Test: 0.0690 | MCC: 0.0139
  Aim: 0.0673 | Cov: 0.1212 | Acc: 0.0617 | AbsT: 0.0000 | Pat: 0/20 >>> BEST <<<
----------------------------------------------------------------------
Ep 02/80 | LR: 6.7e-06 | Train: 0.6548 | Test: 0.0476 | MCC: 0.0562
  Aim: 0.0992 | Cov: 0.0916 | Acc: 0.0916 | AbsT: 0.0881 | Pat: 0/20 >>> BEST <<<
----------------------------------------------------------------------
Ep 03/80 | LR: 1.0e-05 | Train: 0.6375 | Test: 0.0361 | MCC: 0.0863
  Aim: 0.4020 | Cov: 0.3680 | Acc: 0.3680 | AbsT: 0.3448 | Pat: 0/20 >>> BEST <<<
----------------------------------------------------------------------
Ep 04/80 | LR: 1.0e-05 | Train: 0.6255 | Test: 0.0302 | MCC: 0.4043
  Aim: 0.5509 | Cov: 0.5007 | Acc: 0.5007 | AbsT: 0.4653 | Pat: 0/20 >>> BEST <<<
----------------------------

In [181]:
# =====================================================================
# PAPER-STYLE EVALUATION (Table 4 protocol from Zhao et al. 2025)
# =====================================================================
import copy

# 1. Load the best saved model
eval_model = PeptideNetwork(num_classes=21, mask_token_id=32).to(DEVICE)
ckpt = torch.load(best_model_path, map_location=DEVICE, weights_only=False)
eval_model.load_state_dict(ckpt, strict=False)
eval_model.to(DEVICE)
eval_model.eval()

# 2. Run inference on full test set (eval mode, NO TTA)
all_preds = []
all_y = []

with torch.no_grad():
    for data in test_loader:
        input_ids = data['input_ids'].to(DEVICE)
        attention_mask = data['attention_mask'].to(DEVICE)
        pcp_f = data['pcp'].to(DEVICE)
        labels = data['labels'].to(DEVICE)
        all_y.append(labels.cpu().numpy())

        with torch.amp.autocast('cuda'):
            logits = eval_model(input_ids, attention_mask, pcp_f)
        all_preds.append(torch.sigmoid(logits).detach().cpu().numpy())

y_true_full = np.concatenate(all_y)
y_pred_full = np.concatenate(all_preds)

# 3. Construct 5 random 80% test subsets and evaluate each
n_full = len(y_true_full)
n_samp = int(n_full * 0.8)
n_subsets = 5

results = {'precision': [], 'coverage': [], 'accuracy': [], 'absolute_true': [], 'absolute_false': []}

print("=" * 75)
print(f"PAPER-STYLE EVALUATION: {n_subsets} random 80% test subsets (no TTA)")
print(f"Test set: {n_full} samples, subset size: {n_samp}")
print("=" * 75)

for trial in range(n_subsets):
    sidx = np.random.choice(n_full, size=n_samp, replace=False)
    y_true_s = y_true_full[sidx]
    y_pred_s = y_pred_full[sidx]

    thresh = [0.6]*21 #find_optimal_thresholds_accuracy(y_true_s, y_pred_s, rounds=4, step=0.01)
    aim, cov, acc, abst, absf = evaluate(y_pred_s, y_true_s, threshold1=thresh, threshold2=0.3)

    results['precision'].append(aim)
    results['coverage'].append(cov)
    results['accuracy'].append(acc)
    results['absolute_true'].append(abst)
    results['absolute_false'].append(absf)

    print(f"  Subset {trial+1}: acc={acc:.3f} | prec={aim:.3f} | cov={cov:.3f}")

# 4. Print results
print("\n" + "=" * 75)
print("Table 4 format: Five Test Subset Results (Mean +/- sd)")
print("=" * 75)
print(f"{'metric':<20s} {'mean':>8s}   {'± sd':>8s}")
print("-" * 40)
for metric in ['precision', 'coverage', 'accuracy', 'absolute_true', 'absolute_false']:
    vals = np.array(results[metric])
    print(f"{metric:<20s} {np.mean(vals):>8.3f}   ± {np.std(vals):>5.3f}")

# 5. Comparison with paper's MCMFPP
print("\n" + "=" * 75)
print("Comparison with MCMFPP paper (Zhao et al. 2025)")
print("=" * 75)
print(f"{'metric':<20s} {'MCMFPP (paper)':>16s}   {'Ours':>16s}   {'diff':>8s}")
print("-" * 65)
paper = {'precision': 0.775, 'coverage': 0.741, 'accuracy': 0.723, 'absolute_true': 0.663, 'absolute_false': 0.030}
for metric in ['precision', 'coverage', 'accuracy', 'absolute_true', 'absolute_false']:
    ours = np.mean(results[metric])
    theirs = paper[metric]
    diff = ours - theirs
    arrow = "↑" if (diff > 0 and metric != 'absolute_false') or (diff < 0 and metric == 'absolute_false') else "↓"
    print(f"{metric:<20s} {theirs:>16.3f}   {ours:>16.3f}   {diff:>+7.3f} {arrow}")

del eval_model
torch.cuda.empty_cache()

PAPER-STYLE EVALUATION: 5 random 80% test subsets (no TTA)
Test set: 1975 samples, subset size: 1580
  Subset 1: acc=0.709 | prec=0.759 | cov=0.735
  Subset 2: acc=0.708 | prec=0.755 | cov=0.734
  Subset 3: acc=0.707 | prec=0.758 | cov=0.733
  Subset 4: acc=0.702 | prec=0.752 | cov=0.727
  Subset 5: acc=0.718 | prec=0.767 | cov=0.744

Table 4 format: Five Test Subset Results (Mean +/- sd)
metric                   mean       ± sd
----------------------------------------
precision               0.758   ± 0.005
coverage                0.735   ± 0.005
accuracy                0.709   ± 0.005
absolute_true           0.639   ± 0.006
absolute_false          0.032   ± 0.001

Comparison with MCMFPP paper (Zhao et al. 2025)
metric                 MCMFPP (paper)               Ours       diff
-----------------------------------------------------------------
precision                       0.775              0.758    -0.017 ↓
coverage                        0.741              0.735    -0.006 ↓
accur